In [59]:
import pandas as pd
import numpy as np
import os
import re
import json
from datetime import datetime

class FixedDataProcessor:
    def __init__(self):
        self.current_time = "2025-07-19 20:44:21"
        self.current_user = "isaaaaaccccc"
        self.output_dir = 'cleaned_data'
        self.validation_results = {}
        
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Branch code normalization mapping
        self.branch_mapping = {
            'bb': 'BB',
            'cck': 'CCK', 
            'ch': 'CH',
            'changi': 'CH',
            'hg': 'HG',
            'kt': 'KT',
            'katong': 'KT',
            'pr': 'PR'
        }
        
        # Valid levels including Free - UPDATED to include Free
        self.valid_levels = ['Tots', 'Jolly', 'Bubbly', 'Lively', 'Flexi', 'L1', 'L2', 'L3', 'L4', 'Advance', 'Free']
        
        print("="*80)
        print("COMPREHENSIVE DATA PROCESSING SYSTEM - UPDATED FOR FREE CATEGORY")
        print("="*80)
        print(f"Current Date and Time (UTC - YYYY-MM-DD HH:MM:SS formatted): {self.current_time}")
        print(f"Current User's Login: {self.current_user}")
        print(f"Output Directory: {self.output_dir}")
        print(f"Valid Levels: {self.valid_levels}")
        print("UPDATED: Fitness Free → Free (separate from Advance)")
        print()

    def normalize_branch_code(self, branch):
        """Normalize branch codes to match timetable output format"""
        if pd.isna(branch):
            return branch
        
        branch_str = str(branch).lower().strip()
        return self.branch_mapping.get(branch_str, branch_str.upper())

    def process_all_data(self):
        """Main processing pipeline with dynamic data extraction"""
        print("Starting Data Processing Pipeline")
        
        try:
            # Step 1: Load source data
            print("\nStep 1: Loading Source Data")
            source_data = self.load_source_data()
            
            # Step 2: Process coach data  
            print("\nStep 2: Processing Coach Data")
            coaches_df, availability_df = self.process_coach_data(source_data['df2'])
            
            # Step 3: Process enrollment data from CSV
            print("\nStep 3: Processing Enrollment Data from CSV")
            enrollment_counts = self.process_enrollment_data_from_csv(source_data['enrollment_raw'])
            
            # Step 4: Extract branch configuration from Excel
            print("\nStep 4: Extracting Branch Configuration from Excel")
            branch_config = self.extract_branch_config_from_excel(source_data['df1'])
            
            # Step 5: Extract popular timeslots by level
            print("\nStep 5: Extracting Popular Timeslots by Level")
            popular_timeslots = self.extract_popular_timeslots(source_data['df1'])
            
            # Step 6: Save all data
            print("\nStep 6: Saving Processed Data")
            self.save_essential_data({
                'coaches_df': coaches_df,
                'availability_df': availability_df,
                'enrollment_counts': enrollment_counts,
                'branch_config': branch_config,
                'popular_timeslots': popular_timeslots
            })
            
            print(f"\nData Processing Complete")
            
        except Exception as e:
            print(f"Error in data processing: {str(e)}")
            import traceback
            traceback.print_exc()
            raise

    def load_source_data(self):
        """Load all source data files"""
        def read_csv_safe(file_path):
            encodings = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
            for encoding in encodings:
                try:
                    df = pd.read_csv(file_path, encoding=encoding)
                    print(f"   {file_path} loaded with {encoding} encoding")
                    return df
                except UnicodeDecodeError:
                    continue
            raise ValueError(f"Could not read {file_path} with any encoding")
        
        source_data = {}
        
        # Load enrollment data from CSV
        enrollment_path = 'source data/full_retention_rate_v7_cleaned(Sheet1).csv'
        source_data['enrollment_raw'] = read_csv_safe(enrollment_path)
        print(f"   Enrollment data: {len(source_data['enrollment_raw'])} records")
        
        # Load Excel files
        excel_path = 'source data/Completed_Studio Occupany (For Wu Biao).xlsx'
        source_data['df1'] = pd.read_excel(excel_path, sheet_name='Branch Details')
        source_data['df2'] = pd.read_excel(excel_path, sheet_name='Coach Profile ')
        print(f"   Branch details loaded: {source_data['df1'].shape}")
        print(f"   Coach profiles loaded: {source_data['df2'].shape}")
        
        return source_data

    def clean_branch_assignment(self, branch_text):
        """Clean branch assignment text by removing quotes and normalizing"""
        if pd.isna(branch_text):
            return ''
        
        # Convert to string and remove quotes
        branch_str = str(branch_text).strip()
        branch_str = branch_str.replace('"', '').replace("'", '')
        
        return branch_str

    def process_coach_data(self, df2):
        """Process coach data with normalized branch codes - UPDATED: Include Free qualifications"""
        header_row = None
        for idx, row in df2.iterrows():
            if pd.notna(row.iloc[1]) and "Residential Area" in str(row.iloc[1]):
                header_row = idx
                break
        
        if header_row is None:
            raise ValueError("Could not find coach data header row")
        
        data_start = header_row + 1
        data_rows = []
        
        for idx in range(data_start, len(df2)):
            row = df2.iloc[idx]
            if pd.isna(row.iloc[1]) or str(row.iloc[1]).strip() == '' or "Note:" in str(row.iloc[1]):
                break
            data_rows.append(row)
        
        print(f"   Processing {len(data_rows)} coaches")
        
        # Create coaches dataframe with normalized branch codes
        coaches_data = []
        for i, row in enumerate(data_rows):
            # Clean and normalize assigned branch codes
            assigned_branch_raw = self.clean_branch_assignment(row.iloc[2])
            assigned_branches = []
            
            if assigned_branch_raw:
                # Split by comma and normalize each branch
                for branch in assigned_branch_raw.split(','):
                    normalized = self.normalize_branch_code(branch.strip())
                    if normalized:
                        assigned_branches.append(normalized)
            
            assigned_branch_normalized = ', '.join(assigned_branches) if assigned_branches else assigned_branch_raw
            
            coach_info = {
                'coach_id': i + 1,
                'coach_name': str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else '',
                'residential_area': str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else '',
                'assigned_branch': assigned_branch_normalized,
                'position': str(row.iloc[4]).strip() if pd.notna(row.iloc[4]) else '',
                'status': str(row.iloc[5]).strip() if pd.notna(row.iloc[5]) else ''
            }
            
            # Add teaching capabilities
            class_capabilities = [
                ('BearyTots', 9), ('Jolly', 10), ('Bubbly', 11), ('Lively', 12), 
                ('Flexi', 13), ('Level_1', 14), ('Level_2', 15), ('Level_3', 16), 
                ('Level_4', 17)
            ]
            
            for class_type, col_idx in class_capabilities:
                if col_idx < len(row) and pd.notna(row.iloc[col_idx]):
                    coach_info[class_type] = str(row.iloc[col_idx]).strip().lower() == 'yes'
                else:
                    coach_info[class_type] = False
            
            # UPDATED: Handle Advance and Free as separate levels
            advance_col = 19  # Advance column
            free_col = 20     # Free column (if exists)
            
            # Advance level
            if advance_col < len(row) and pd.notna(row.iloc[advance_col]):
                coach_info['Advance'] = str(row.iloc[advance_col]).strip().lower() == 'yes'
            else:
                coach_info['Advance'] = False
            
            # Free level (separate from Advance) - UPDATED
            if free_col < len(row) and pd.notna(row.iloc[free_col]):
                coach_info['Free'] = str(row.iloc[free_col]).strip().lower() == 'yes'
            else:
                # Fallback: If no separate Free column, assume coaches qualified for Advance can also do Free
                coach_info['Free'] = coach_info['Advance']
                print(f"   Coach {i+1}: No separate Free column, using Advance qualification ({coach_info['Advance']}) for Free")
            
            coaches_data.append(coach_info)
        
        coaches_df = pd.DataFrame(coaches_data)
        
        # Create availability dataframe
        availability_data = []
        for _, coach in coaches_df.iterrows():
            row = data_rows[coach['coach_id'] - 1]
            
            full_off = str(row.iloc[6]).strip() if pd.notna(row.iloc[6]) else ''
            half_off = str(row.iloc[7]).strip() if pd.notna(row.iloc[7]) else ''
            restricted_times = str(row.iloc[8]).strip() if pd.notna(row.iloc[8]) else ''
            
            coach_availability = self.parse_coach_availability(
                coach['coach_id'], full_off, half_off, restricted_times
            )
            availability_data.extend(coach_availability)
        
        availability_df = pd.DataFrame(availability_data)
        
        print(f"   Coaches processed: {len(coaches_df)}")
        print(f"   Availability records: {len(availability_df)}")
        print(f"   Branch codes normalized: {sorted(set([b for coach_branches in coaches_df['assigned_branch'] for b in coach_branches.split(', ') if b]))}")
        
        # Show Free qualification summary
        free_qualified_coaches = len(coaches_df[coaches_df['Free'] == True])
        advance_qualified_coaches = len(coaches_df[coaches_df['Advance'] == True])
        print(f"   Coaches qualified for Free: {free_qualified_coaches}")
        print(f"   Coaches qualified for Advance: {advance_qualified_coaches}")
        
        return coaches_df, availability_df

    def parse_coach_availability(self, coach_id, full_off, half_off, restricted_times):
        """Parse coach availability"""
        availability = []
        days = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']
        periods = ['am', 'pm']
        
        full_off_days = self.parse_day_restrictions(full_off)
        half_off_periods = self.parse_period_restrictions(half_off)
        restricted_periods = self.parse_restricted_times(restricted_times)
        
        availability_id = 1
        for day in days:
            for period in periods:
                available = True
                restriction_reason = ''
                
                if day in full_off_days:
                    available = False
                    restriction_reason = 'full_day_off'
                elif day in half_off_periods and period in half_off_periods[day]:
                    available = False
                    restriction_reason = 'half_day_off'
                elif day in restricted_periods and period in restricted_periods[day]:
                    available = False
                    restriction_reason = 'time_restriction'
                
                availability.append({
                    'availability_id': availability_id,
                    'coach_id': coach_id,
                    'day': day,
                    'period': period,
                    'available': available,
                    'restriction_reason': restriction_reason
                })
                availability_id += 1
        
        return availability

    def parse_day_restrictions(self, restriction_text):
        """Parse full day restrictions"""
        if not restriction_text:
            return set()
        
        days = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']
        restricted_days = set()
        text = restriction_text.lower()
        
        for day in days:
            if day.lower() in text:
                restricted_days.add(day)
        
        return restricted_days

    def parse_period_restrictions(self, restriction_text):
        """Parse half day restrictions"""
        if not restriction_text:
            return {}
        
        days = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']
        restricted_periods = {}
        text = restriction_text.lower()
        
        for day in days:
            if f"{day.lower()} am" in text:
                restricted_periods[day] = restricted_periods.get(day, []) + ['am']
            if f"{day.lower()} pm" in text:
                restricted_periods[day] = restricted_periods.get(day, []) + ['pm']
        
        return restricted_periods

    def parse_restricted_times(self, restricted_times):
        """Parse time restrictions"""
        if not restricted_times:
            return {}
        
        days = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']
        restricted_periods = {}
        text = restricted_times.lower()
        
        # Handle ranges like "Mon - Thu am pm"
        range_pattern = r'(\w+)\s*-\s*(\w+)\s+(am\s+pm|am|pm)'
        ranges = re.findall(range_pattern, text)
        
        for start_day, end_day, period in ranges:
            start_idx = next((i for i, d in enumerate(days) if d.lower().startswith(start_day.lower())), -1)
            end_idx = next((i for i, d in enumerate(days) if d.lower().startswith(end_day.lower())), -1)
            
            if start_idx != -1 and end_idx != -1:
                for i in range(start_idx, end_idx + 1):
                    day = days[i]
                    periods = ['am', 'pm'] if 'am pm' in period else [period.strip()]
                    restricted_periods[day] = restricted_periods.get(day, []) + periods
        
        return restricted_periods

    def process_enrollment_data_from_csv(self, enrollment_raw):
        """Process enrollment data - UPDATED: Properly handle fitness free as Free"""
        print("   Processing enrollment data from CSV file")
        
        # Examine the CSV structure
        print(f"   CSV columns: {list(enrollment_raw.columns)}")
        print(f"   CSV shape: {enrollment_raw.shape}")
        
        # DEBUG: Check unique values in Year and Term columns
        if 'Year' in enrollment_raw.columns:
            print(f"   Unique Years: {sorted(enrollment_raw['Year'].unique())}")
            print(f"   Year data types: {enrollment_raw['Year'].dtype}")
        else:
            print("   WARNING: 'Year' column not found!")
            print(f"   Available columns: {list(enrollment_raw.columns)}")
        
        if 'Term' in enrollment_raw.columns:
            print(f"   Unique Terms: {sorted(enrollment_raw['Term'].unique())}")
            print(f"   Term data types: {enrollment_raw['Term'].dtype}")
        else:
            print("   WARNING: 'Term' column not found!")
        
        # Check for alternative column names
        year_candidates = [col for col in enrollment_raw.columns if 'year' in col.lower()]
        term_candidates = [col for col in enrollment_raw.columns if 'term' in col.lower()]
        
        print(f"   Year candidate columns: {year_candidates}")
        print(f"   Term candidate columns: {term_candidates}")
        
        # Filter for 2025 T1 only - with flexible matching
        print("   Filtering for 2025 T1 only...")
        
        # Try different filtering approaches
        filtered_data = None
        
        # Approach 1: Standard filtering
        if 'Year' in enrollment_raw.columns and 'Term' in enrollment_raw.columns:
            try:
                # Try exact match first
                filtered_data = enrollment_raw[
                    (enrollment_raw['Year'] == 2025) & 
                    (enrollment_raw['Term'] == 'T1')
                ]
                print(f"   Exact match filtering: {len(filtered_data)} records")
                
                # If no results, try alternative matches
                if len(filtered_data) == 0:
                    # Try string conversion for Year
                    filtered_data = enrollment_raw[
                        (enrollment_raw['Year'].astype(str).str.contains('2025', na=False)) & 
                        (enrollment_raw['Term'].astype(str).str.upper() == 'T1')
                    ]
                    print(f"   String-based filtering: {len(filtered_data)} records")
                
                # If still no results, try case-insensitive term matching
                if len(filtered_data) == 0:
                    filtered_data = enrollment_raw[
                        (enrollment_raw['Year'] == 2025) & 
                        (enrollment_raw['Term'].astype(str).str.upper().isin(['T1', 'TERM 1', 'TERM1']))
                    ]
                    print(f"   Case-insensitive term filtering: {len(filtered_data)} records")
                    
            except Exception as e:
                print(f"   Error in filtering: {str(e)}")
                filtered_data = None
        
        # Approach 2: If standard columns don't work, try alternatives
        if filtered_data is None or len(filtered_data) == 0:
            print("   Standard filtering failed, trying alternative approaches...")
            
            # Check if we should use all data
            total_records = len(enrollment_raw)
            print(f"   Total records available: {total_records}")
            
            # Sample some data to understand the structure
            print("   Sample data (first 5 rows):")
            for col in enrollment_raw.columns:
                print(f"     {col}: {enrollment_raw[col].head().tolist()}")
            
            # For now, use all data if filtering fails
            print("   WARNING: Using all enrollment data since 2025 T1 filtering failed")
            filtered_data = enrollment_raw.copy()
        
        print(f"   Filtered data: {len(filtered_data)} records (from {len(enrollment_raw)} total)")
        
        # Use the correct columns based on the CSV structure
        branch_col = 'Branch'
        level_col = 'Level Category Base'  # This has the cleaned level categories
        
        # Check if these columns exist
        if branch_col not in filtered_data.columns:
            print(f"   ERROR: '{branch_col}' column not found!")
            print(f"   Available columns: {list(filtered_data.columns)}")
            # Try to find alternative branch column
            branch_candidates = [col for col in filtered_data.columns if 'branch' in col.lower()]
            if branch_candidates:
                branch_col = branch_candidates[0]
                print(f"   Using alternative branch column: {branch_col}")
            else:
                print("   No branch column found - cannot process enrollment data")
                return pd.DataFrame(columns=['Branch', 'Level Category Base', 'Count'])
        
        if level_col not in filtered_data.columns:
            print(f"   ERROR: '{level_col}' column not found!")
            print(f"   Available columns: {list(filtered_data.columns)}")
            # Try to find alternative level column
            level_candidates = [col for col in filtered_data.columns if 'level' in col.lower()]
            if level_candidates:
                level_col = level_candidates[0]
                print(f"   Using alternative level column: {level_col}")
            else:
                print("   No level column found - cannot process enrollment data")
                return pd.DataFrame(columns=['Branch', 'Level Category Base', 'Count'])
        
        print(f"   Using columns: Branch={branch_col}, Level={level_col}")
        print("   Counting students per branch-level combination...")
        
        # Count students by grouping branch and level
        try:
            enrollment_counts = filtered_data.groupby([branch_col, level_col]).size().reset_index(name='Count')
            print(f"   Grouping successful: {len(enrollment_counts)} unique combinations")
        except Exception as e:
            print(f"   Error in groupby: {str(e)}")
            return pd.DataFrame(columns=['Branch', 'Level Category Base', 'Count'])
        
        # Normalize branch codes and clean levels - UPDATED: Handle Free properly
        enrollment_list = []
        skipped_count = 0
        free_count = 0
        
        for _, row in enrollment_counts.iterrows():
            branch_raw = str(row[branch_col]).strip()
            level_raw = str(row[level_col]).strip()
            count = int(row['Count'])
            
            # Skip invalid entries
            if branch_raw == 'nan' or level_raw == 'nan' or count <= 0:
                print(f"   Skipping invalid entry: {branch_raw}, {level_raw}, {count}")
                skipped_count += 1
                continue
            
            # Normalize branch code
            branch_normalized = self.normalize_branch_code(branch_raw)
            
            # Clean level category - UPDATED: Handle Free properly
            level_normalized = self.clean_level_category(level_raw)
            
            # Skip if level is not valid after cleaning
            if level_normalized not in self.valid_levels:
                print(f"   ❌ SKIPPING INVALID LEVEL: {branch_raw}, '{level_raw}' -> '{level_normalized}' (not in valid levels)")
                skipped_count += 1
                continue
            
            # Track Free students
            if level_normalized == 'Free':
                free_count += count
                print(f"   ✅ FREE STUDENT: {branch_raw} -> {branch_normalized}, '{level_raw}' -> {level_normalized}, Count: {count}")
            else:
                print(f"   ✅ Processing: {branch_raw} -> {branch_normalized}, {level_raw} -> {level_normalized}, Count: {count}")
            
            enrollment_list.append({
                'Branch': branch_normalized,
                'Level Category Base': level_normalized,
                'Count': count
            })
        
        enrollment_counts_df = pd.DataFrame(enrollment_list)
        
        print(f"   Processed {len(enrollment_counts_df)} enrollment combinations")
        print(f"   Skipped {skipped_count} invalid/non-gymnastics entries")
        print(f"   FREE STUDENTS FOUND: {free_count} students identified as 'Free' level")
        
        if len(enrollment_counts_df) > 0:
            print(f"   Valid Branches: {sorted(enrollment_counts_df['Branch'].unique())}")
            print(f"   Valid Levels: {sorted(enrollment_counts_df['Level Category Base'].unique())}")
            print(f"   Total gymnastics students: {enrollment_counts_df['Count'].sum()}")
            
            # Show Free vs Advance breakdown
            advance_students = enrollment_counts_df[enrollment_counts_df['Level Category Base'] == 'Advance']['Count'].sum()
            free_students = enrollment_counts_df[enrollment_counts_df['Level Category Base'] == 'Free']['Count'].sum()
            print(f"   Advance students: {advance_students}")
            print(f"   Free students: {free_students}")
        else:
            print("   WARNING: No valid enrollment data processed!")
        
        return enrollment_counts_df

    def clean_level_category(self, level):
        """Clean level categories - UPDATED: Properly handle fitness free as Free"""
        if pd.isna(level) or level == '':
            return level
        
        level_str = str(level).strip()
        level_lower = level_str.lower()
        
        print(f"     Level cleaning: '{level_str}' (lowercase: '{level_lower}')")
        
        # PRIORITY 1: FITNESS FREE DETECTION - Check for "fitness free" or "free" in the level
        if 'fitness free' in level_lower or level_lower == 'fitness free':
            print(f"     ✅ FITNESS FREE DETECTED: '{level_str}' -> returning 'Free'")
            return 'Free'
        
        # PRIORITY 2: General "free" detection (but not "fitness" alone)
        if 'free' in level_lower and 'fitness' not in level_lower:
            print(f"     ✅ FREE DETECTED in '{level_str}' -> returning 'Free'")
            return 'Free'
        
        # PRIORITY 3: Standalone "free" level
        if level_lower == 'free':
            print(f"     ✅ STANDALONE FREE: '{level_str}' -> returning 'Free'")
            return 'Free'
        
        # PRIORITY 4: REJECT general fitness levels (but NOT fitness free)
        if 'fitness' in level_lower and 'free' not in level_lower:
            # Check if it's a fitness level combined with regular gymnastics level
            fitness_pattern = re.search(r'fitness\s+(l\d+|level\s*\d+)', level_lower)
            if fitness_pattern:
                # Extract the gymnastics level from fitness level
                gymnastics_part = fitness_pattern.group(1)
                print(f"     ✅ FITNESS + GYMNASTICS: '{level_str}' contains '{gymnastics_part}' -> extracting gymnastics level")
                
                # Clean the extracted gymnastics level
                cleaned_gymnastics = self.clean_level_category(gymnastics_part)
                if cleaned_gymnastics in self.valid_levels:
                    print(f"     ✅ EXTRACTED LEVEL: '{level_str}' -> '{cleaned_gymnastics}'")
                    return cleaned_gymnastics
            
            print(f"     ❌ GENERAL FITNESS REJECTED: '{level_str}' -> not a gymnastics level")
            return 'INVALID_FITNESS'
        
        # PRIORITY 5: L5 or Level 5 conversion to Advance
        if re.search(r'\bl5\b', level_lower) or re.search(r'\blevel\s*5\b', level_lower):
            print(f"     ✅ L5 DETECTED in '{level_str}' -> returning 'Advance'")
            return 'Advance'
        
        # PRIORITY 6: Level+ conversion (L3+ -> L4, L2+ -> L3, etc.)
        plus_pattern = re.search(r'\b(l|level\s*)(\d+)\+', level_lower)
        if plus_pattern:
            level_num = int(plus_pattern.group(2))
            next_level_num = level_num + 1
            
            # Map to next level, but cap at L4 (since L5 becomes Advance)
            if next_level_num >= 5:
                result = 'Advance'
            else:
                result = f'L{next_level_num}'
            
            print(f"     ✅ LEVEL+ DETECTED: '{level_str}' (L{level_num}+) -> returning '{result}'")
            return result
        
        # PRIORITY 7: Direct mapping for exact matches - ONLY VALID GYMNASTICS LEVELS
        level_mapping = {
            'bearytots': 'Tots',
            'tots': 'Tots',
            'advance': 'Advance',
            'advanced': 'Advance',
            'adv': 'Advance',
            'l1': 'L1', 'l2': 'L2', 'l3': 'L3', 'l4': 'L4',
            'level 1': 'L1', 'level 2': 'L2', 'level 3': 'L3', 'level 4': 'L4',
            'level1': 'L1', 'level2': 'L2', 'level3': 'L3', 'level4': 'L4',
            'jolly': 'Jolly', 'bubbly': 'Bubbly', 'lively': 'Lively', 'flexi': 'Flexi'
        }
        
        # Try exact match first (case-insensitive)
        exact_match = level_mapping.get(level_lower)
        if exact_match:
            print(f"     ✅ EXACT MATCH: '{level_str}' -> '{exact_match}'")
            return exact_match
        
        # PRIORITY 8: Pattern-based matching for complex combinations
        # Check for L2/L3, L3/L4 patterns (but NO fitness unless it's fitness free)
        if '/' in level_str and ('free' in level_lower or 'fitness' not in level_lower):
            parts = [part.strip() for part in level_str.split('/')]
            normalized_parts = []
            has_invalid = False
            
            for part in parts:
                part_lower = part.lower()
                
                # Handle fitness free specifically
                if 'fitness free' in part_lower or part_lower == 'free':
                    normalized_parts.append('Free')
                # Reject other fitness
                elif 'fitness' in part_lower:
                    has_invalid = True
                    break
                # Apply L5 rule to each part
                elif re.search(r'\bl5\b', part_lower) or re.search(r'\blevel\s*5\b', part_lower):
                    normalized_parts.append('Advance')
                # Apply Level+ rule to each part
                elif re.search(r'\b(l|level\s*)(\d+)\+', part_lower):
                    plus_match = re.search(r'\b(l|level\s*)(\d+)\+', part_lower)
                    level_num = int(plus_match.group(2))
                    next_level_num = level_num + 1
                    if next_level_num >= 5:
                        normalized_parts.append('Advance')
                    else:
                        normalized_parts.append(f'L{next_level_num}')
                elif part_lower in level_mapping:
                    normalized_parts.append(level_mapping[part_lower])
                elif part_lower in ['adv', 'advance', 'advanced']:
                    normalized_parts.append('Advance')
                else:
                    # If we can't map this part, mark as invalid
                    has_invalid = True
                    break
            
            if not has_invalid and normalized_parts:
                result = '/'.join(normalized_parts)
                print(f"     ✅ PATTERN MATCH: '{level_str}' -> '{result}'")
                return result
        
        # Check for "+" patterns like "Lively + Flexi" (but NO fitness unless fitness free)
        if '+' in level_str and not re.search(r'\b(l|level\s*)(\d+)\+', level_lower):
            parts = [part.strip() for part in level_str.split('+')]
            normalized_parts = []
            has_invalid = False
            
            for part in parts:
                part_lower = part.lower()
                
                # Handle fitness free specifically
                if 'fitness free' in part_lower or part_lower == 'free':
                    normalized_parts.append('Free')
                # Reject other fitness
                elif 'fitness' in part_lower:
                    has_invalid = True
                    break
                # Apply L5 rule to each part
                elif re.search(r'\bl5\b', part_lower) or re.search(r'\blevel\s*5\b', part_lower):
                    normalized_parts.append('Advance')
                elif part_lower in level_mapping:
                    normalized_parts.append(level_mapping[part_lower])
                else:
                    has_invalid = True
                    break
            
            if not has_invalid and normalized_parts:
                result = ' + '.join(normalized_parts)
                print(f"     ✅ PLUS PATTERN: '{level_str}' -> '{result}'")
                return result
        
        # PRIORITY 9: Fallback - mark as invalid if no valid match found
        print(f"     ❌ INVALID LEVEL: '{level_str}' -> not a valid gymnastics level")
        return 'INVALID_LEVEL'

    def extract_branch_config_from_excel(self, df1):
        """Extract branch configuration dynamically from Excel file"""
        print("   Extracting branch configuration from Excel file")
        
        # Search for the capacity table
        capacity_limits = {}
        found_capacity_section = False
        
        for idx, row in df1.iterrows():
            # Convert row to text for searching
            row_text = ' '.join([str(cell) for cell in row if pd.notna(cell)]).lower()
            
            # Look for the capacity section header
            if "maximum classes" in row_text and ("slot" in row_text or "hour" in row_text):
                print(f"   Found capacity section at row {idx}")
                found_capacity_section = True
                
                # Look for the data rows following this header
                for data_idx in range(idx + 1, min(idx + 15, len(df1))):
                    if data_idx >= len(df1):
                        break
                    
                    data_row = df1.iloc[data_idx]
                    
                    # Check each cell in the row for branch-capacity pairs
                    for col_idx in range(len(data_row)):
                        cell_value = str(data_row.iloc[col_idx]).strip() if pd.notna(data_row.iloc[col_idx]) else ''
                        
                        # Look for branch codes
                        if cell_value.upper() in ['BB', 'CCK', 'CH', 'HG', 'KT', 'PR']:
                            branch_code = cell_value.upper()
                            
                            # Look for capacity value in next column
                            if col_idx + 1 < len(data_row):
                                capacity_cell = str(data_row.iloc[col_idx + 1]).strip()
                                
                                # Extract number from capacity cell
                                capacity_match = re.search(r'(\d+)', capacity_cell)
                                if capacity_match:
                                    capacity_num = int(capacity_match.group(1))
                                    capacity_limits[branch_code] = capacity_num
                                    print(f"   Found: {branch_code} = {capacity_num} classes per slot")
                
                break
        
        if not found_capacity_section:
            print("   Warning: Could not find capacity section in Excel file")
            print("   Searching for any branch-capacity data...")
            
            # Alternative search method
            for idx, row in df1.iterrows():
                for col_idx in range(len(row)):
                    cell_value = str(row.iloc[col_idx]).strip() if pd.notna(row.iloc[col_idx]) else ''
                    
                    # Look for patterns like "BB" followed by numbers
                    if cell_value.upper() in ['BB', 'CCK', 'CH', 'HG', 'KT', 'PR']:
                        branch_code = cell_value.upper()
                        
                        # Check adjacent cells for numbers
                        for offset in [1, -1]:
                            check_col = col_idx + offset
                            if 0 <= check_col < len(row):
                                adjacent_cell = str(row.iloc[check_col]).strip()
                                capacity_match = re.search(r'(\d+)', adjacent_cell)
                                if capacity_match and branch_code not in capacity_limits:
                                    capacity_num = int(capacity_match.group(1))
                                    if 1 <= capacity_num <= 20:  # Reasonable range check
                                        capacity_limits[branch_code] = capacity_num
                                        print(f"   Found: {branch_code} = {capacity_num} classes per slot")
        
        # Create branch config dataframe
        if capacity_limits:
            branch_config_data = []
            for branch_code, capacity in capacity_limits.items():
                branch_config_data.append({
                    'branch': branch_code,
                    'max_classes_per_slot': capacity
                })
            
            branch_config_df = pd.DataFrame(branch_config_data)
            print(f"   Successfully extracted {len(branch_config_df)} branch configurations")
        else:
            print("   Error: No branch capacity data found in Excel file")
            print("   Creating empty branch config")
            branch_config_df = pd.DataFrame(columns=['branch', 'max_classes_per_slot'])
        
        return branch_config_df

    def extract_popular_timeslots(self, df1):
        """Extract popular timeslots ONLY for valid gymnastics levels - UPDATED: Include Free"""
        print("   Extracting popular timeslots by level from Excel (gymnastics levels including Free)")
        
        timeslot_data = []
        
        # Step 1: Find the timing table header "Popular Class Type at Specific Timing"
        timing_table_start = None
        for idx, row in df1.iterrows():
            row_text = ' '.join([str(cell) for cell in row if pd.notna(cell)])
            if "Popular Class Type at Specific Timing" in row_text:
                print(f"   Found timing table header at row {idx}")
                timing_table_start = idx
                break
        
        if timing_table_start is None:
            print("   Error: Could not find timing table")
            return pd.DataFrame(columns=['time_slot', 'day', 'level'])
        
        # Step 2: Find the actual timing data section (look for "Timing" in first column)
        timing_data_start = None
        for check_idx in range(timing_table_start, min(timing_table_start + 10, len(df1))):
            row = df1.iloc[check_idx]
            first_cell = str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else ''  # Column 3 based on debug
            
            if first_cell.lower() == "timing":
                timing_data_start = check_idx + 1  # Data starts next row
                print(f"   Found 'Timing' section at row {check_idx}, data starts at {timing_data_start}")
                break
        
        if timing_data_start is None:
            print("   Error: Could not find timing data section")
            return pd.DataFrame(columns=['time_slot', 'day', 'level'])
        
        # Step 3: Define day mappings based on the debug output
        day_columns = {
            'MON': 4, 'TUE': 5, 'WED': 6, 'THU': 7, 
            'FRI': 8, 'SAT': 9, 'SUN': 10
        }
        
        print(f"   Using day column mapping: {day_columns}")
        print(f"   Parsing timing data starting from row {timing_data_start}")
        
        # Step 4: Extract time slots and levels from the timing data rows
        invalid_count = 0
        
        for row_idx in range(timing_data_start, min(timing_data_start + 10, len(df1))):
            if row_idx >= len(df1):
                break
                
            row = df1.iloc[row_idx]
            
            # Time slot is in column 3 (based on debug output)
            time_cell = str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else ''
            
            # Check if this cell contains a time slot
            if time_cell and ('am' in time_cell.lower() or 'pm' in time_cell.lower()):
                # Clean up the time slot text
                time_slot = time_cell.replace('\n', ' ').strip()
                print(f"   Processing time slot: '{time_slot}'")
                
                # Extract levels for each day
                for day, col_idx in day_columns.items():
                    if col_idx < len(row):
                        cell_content = str(row.iloc[col_idx]).strip() if pd.notna(row.iloc[col_idx]) else ''
                        
                        if cell_content and cell_content.lower() not in ['nan', 'closed', '']:
                            print(f"     {day} (col {col_idx}): '{cell_content}'")
                            
                            # Split levels by newlines
                            levels = cell_content.split('\n')
                            
                            for level in levels:
                                level = level.strip()
                                if level and len(level) > 1:
                                    # Normalize level names - INCLUDE FREE AS VALID
                                    level_normalized = self.normalize_level_name(level)
                                    if level_normalized and level_normalized in self.valid_levels:
                                        timeslot_data.append({
                                            'time_slot': self.normalize_time_slot(time_slot),
                                            'day': day,
                                            'level': level_normalized
                                        })
                                        if level_normalized == 'Free':
                                            print(f"       ✅ Added FREE: {day} {level_normalized} at {time_slot}")
                                        else:
                                            print(f"       ✅ Added: {day} {level_normalized} at {time_slot}")
                                    elif level_normalized:
                                        print(f"       ❌ Skipped invalid level: {level} -> {level_normalized}")
                                        invalid_count += 1
        
        # Create DataFrame
        popular_timeslots_df = pd.DataFrame(timeslot_data)
        
        print(f"   Extracted {len(popular_timeslots_df)} valid gymnastics timeslot records")
        print(f"   Skipped {invalid_count} non-gymnastics level entries")
        if len(popular_timeslots_df) > 0:
            unique_slots = sorted(popular_timeslots_df['time_slot'].unique())
            unique_levels = sorted(popular_timeslots_df['level'].unique())
            print(f"   Time slots found: {unique_slots}")
            print(f"   Valid gymnastics levels found: {unique_levels}")
            
            # Show Free timeslots specifically
            free_timeslots = len(popular_timeslots_df[popular_timeslots_df['level'] == 'Free'])
            advance_timeslots = len(popular_timeslots_df[popular_timeslots_df['level'] == 'Advance'])
            print(f"   Free timeslots: {free_timeslots}")
            print(f"   Advance timeslots: {advance_timeslots}")
        else:
            print("   Warning: No valid gymnastics timeslot data extracted")
        
        return popular_timeslots_df

    def normalize_time_slot(self, time_text):
        """Normalize time slot text to standard format"""
        if not time_text:
            return time_text
        
        # Convert formats like "8.30am to 12.30pm" to "08:30-12:30"
        time_pattern = r'(\d+)\.?(\d*)\s*(am|pm)\s*(?:to|-)\s*(\d+)\.?(\d*)\s*(am|pm)'
        match = re.search(time_pattern, time_text.lower())
        
        if match:
            start_hour, start_min, start_period, end_hour, end_min, end_period = match.groups()
            
            # Convert to 24-hour format
            start_hour = int(start_hour)
            end_hour = int(end_hour)
            start_min = start_min if start_min else '00'
            end_min = end_min if end_min else '00'
            
            if start_period == 'pm' and start_hour != 12:
                start_hour += 12
            elif start_period == 'am' and start_hour == 12:
                start_hour = 0
                
            if end_period == 'pm' and end_hour != 12:
                end_hour += 12
            elif end_period == 'am' and end_hour == 12:
                end_hour = 0
            
            return f"{start_hour:02d}:{start_min}-{end_hour:02d}:{end_min}"
        
        return time_text

    def normalize_level_name(self, level):
        """Normalize level names - UPDATED: Include Free as valid gymnastics level"""
        if not level:
            return None
        
        level = level.strip()
        level_lower = level.lower()
        
        print(f"       Level normalization: '{level}' (lowercase: '{level_lower}')")
        
        # PRIORITY 1: FITNESS FREE DETECTION - specific check for fitness free
        if 'fitness free' in level_lower or level_lower == 'fitness free':
            print(f"       ✅ FITNESS FREE DETECTED: '{level}' -> returning 'Free'")
            return 'Free'
        
        # PRIORITY 2: General FREE detection (but not general fitness)
        if 'free' in level_lower and ('fitness' not in level_lower or 'fitness free' in level_lower):
            print(f"       ✅ FREE DETECTED in '{level}' -> returning 'Free'")
            return 'Free'
        
        # PRIORITY 3: REJECT other fitness levels
        if 'fitness' in level_lower and 'free' not in level_lower:
            print(f"       ❌ FITNESS REJECTED: '{level}' -> not a gymnastics level")
            return 'INVALID_FITNESS'
        
        # PRIORITY 4: L5 or Level 5 conversion to Advance
        if re.search(r'\bl5\b', level_lower) or re.search(r'\blevel\s*5\b', level_lower):
            print(f"       ✅ L5 DETECTED in '{level}' -> returning 'Advance'")
            return 'Advance'
        
        # PRIORITY 5: Level+ conversion (L3+ -> L4, L2+ -> L3, etc.)
        plus_pattern = re.search(r'\b(l|level\s*)(\d+)\+', level_lower)
        if plus_pattern:
            level_num = int(plus_pattern.group(2))
            next_level_num = level_num + 1
            
            # Map to next level, but cap at L4 (since L5 becomes Advance)
            if next_level_num >= 5:
                result = 'Advance'
            else:
                result = f'L{next_level_num}'
            
            print(f"       ✅ LEVEL+ DETECTED: '{level}' (L{level_num}+) -> returning '{result}'")
            return result
        
        # PRIORITY 6: Direct mapping for exact matches - INCLUDING FREE
        level_mapping = {
            'tots': 'Tots',
            'bearytots': 'Tots',
            'jolly': 'Jolly',
            'bubbly': 'Bubbly', 
            'lively': 'Lively',
            'flexi': 'Flexi',
            'level 1': 'L1', 'level1': 'L1', 'l1': 'L1',
            'level 2': 'L2', 'level2': 'L2', 'l2': 'L2',
            'level 3': 'L3', 'level3': 'L3', 'l3': 'L3',
            'level 4': 'L4', 'level4': 'L4', 'l4': 'L4',
            'adv': 'Advance', 'advance': 'Advance', 'advanced': 'Advance',
            'free': 'Free'  # UPDATED: Add free mapping
        }
        
        # Try exact match (case-insensitive)
        exact_match = level_mapping.get(level_lower)
        if exact_match:
            print(f"       ✅ EXACT MATCH: '{level}' -> '{exact_match}'")
            return exact_match
        
        # PRIORITY 7: Return invalid if no match
        print(f"       ❌ INVALID LEVEL: '{level}' -> not a valid gymnastics level")
        return 'INVALID_LEVEL'

    def save_essential_data(self, processed_data):
        """Save all processed data files - UPDATED: Include Free category information"""
        files_saved = []
        
        # Save all data files including the new popular_timeslots
        data_files = ['coaches_df', 'availability_df', 'enrollment_counts', 'branch_config', 'popular_timeslots']
        
        for data_name in data_files:
            df = processed_data.get(data_name)
            if df is not None:
                file_path = os.path.join(self.output_dir, f"{data_name}.csv")
                df.to_csv(file_path, index=False, encoding='utf-8')
                files_saved.append(f"{data_name}.csv")
                print(f"   {data_name}: {len(df)} records -> {file_path}")
            else:
                print(f"   Warning: {data_name} is None, skipping save")
        
        print(f"\nFILES SUMMARY:")
        print(f"   coaches_df.csv - Coach profiles with Free qualifications")
        print(f"   availability_df.csv - Coach availability by day/period")
        print(f"   enrollment_counts.csv - GYMNASTICS + FREE (fitness entries processed correctly)")
        print(f"   branch_config.csv - Branch capacity limits from Excel")
        print(f"   popular_timeslots.csv - Popular timeslots for GYMNASTICS + FREE")
        print(f"   Valid gymnastics levels: {self.valid_levels}")
        
        # Show specific Free category statistics
        enrollment_counts = processed_data.get('enrollment_counts')
        if enrollment_counts is not None and len(enrollment_counts) > 0:
            free_students = enrollment_counts[enrollment_counts['Level Category Base'] == 'Free']['Count'].sum()
            advance_students = enrollment_counts[enrollment_counts['Level Category Base'] == 'Advance']['Count'].sum()
            total_students = enrollment_counts['Count'].sum()
            
            print(f"\nFREE CATEGORY STATISTICS:")
            print(f"   Free students: {free_students}")
            print(f"   Advance students: {advance_students}")
            print(f"   Total students: {total_students}")
            print(f"   Free percentage: {(free_students/total_students*100):.1f}% of total students")
        
        coaches_df = processed_data.get('coaches_df')
        if coaches_df is not None:
            free_qualified = len(coaches_df[coaches_df['Free'] == True])
            advance_qualified = len(coaches_df[coaches_df['Advance'] == True])
            total_coaches = len(coaches_df)
            
            print(f"\nCOACH QUALIFICATIONS:")
            print(f"   Free qualified coaches: {free_qualified}/{total_coaches}")
            print(f"   Advance qualified coaches: {advance_qualified}/{total_coaches}")
        
        print(f"   Ready for gymnastics timetabling with FREE as separate category from ADVANCE")
        
        return files_saved

# Main execution
if __name__ == "__main__":
    processor = FixedDataProcessor()
    processor.process_all_data()

COMPREHENSIVE DATA PROCESSING SYSTEM - UPDATED FOR FREE CATEGORY
Current Date and Time (UTC - YYYY-MM-DD HH:MM:SS formatted): 2025-07-19 20:44:21
Current User's Login: isaaaaaccccc
Output Directory: cleaned_data
Valid Levels: ['Tots', 'Jolly', 'Bubbly', 'Lively', 'Flexi', 'L1', 'L2', 'L3', 'L4', 'Advance', 'Free']
UPDATED: Fitness Free → Free (separate from Advance)

Starting Data Processing Pipeline

Step 1: Loading Source Data
   source data/full_retention_rate_v7_cleaned(Sheet1).csv loaded with latin-1 encoding
   Enrollment data: 29250 records
   Branch details loaded: (29, 16)
   Coach profiles loaded: (45, 21)

Step 2: Processing Coach Data
   Processing 41 coaches
   Coach 1: No separate Free column, using Advance qualification (False) for Free
   Coach 2: No separate Free column, using Advance qualification (False) for Free
   Coach 3: No separate Free column, using Advance qualification (False) for Free
   Coach 4: No separate Free column, using Advance qualification (False) f

In [47]:
# df1.to_csv('df1.csv', index=False, encoding='utf-8')

In [48]:
# df1

In [49]:
# schedule_df.to_csv('cleaned data/schedule_df.csv', index=False, encoding='utf-8')
# capacity_df.to_csv('cleaned data/capacity_df.csv', index=False, encoding='utf-8')

In [50]:
# import pandas as pd
# import numpy as np

# def clean_coach_dataframe(df2):
#     """
#     Clean and process coach dataframe into two separate, non-overlapping dataframes
#     """
    
#     # Find the header row (contains "Residential Area", "Coach Name", etc.)
#     header_row = None
#     for idx, row in df2.iterrows():
#         if pd.notna(row.iloc[1]) and "Residential Area" in str(row.iloc[1]):
#             header_row = idx
#             break
    
#     if header_row is None:
#         raise ValueError("Could not find header row")
    
#     # Extract data rows (skip notes at the end)
#     data_start = header_row + 1
#     data_rows = []
    
#     for idx in range(data_start, len(df2)):
#         row = df2.iloc[idx]
#         # Stop when we hit empty rows or notes
#         if pd.isna(row.iloc[1]) or str(row.iloc[1]).strip() == '' or "Note:" in str(row.iloc[1]):
#             break
#         data_rows.append(row)
    
#     # 1. COACHES DATAFRAME - Basic info + capabilities
#     coaches_data = []
    
#     for row in data_rows:
#         coach_info = {
#             'coach_id': len(coaches_data) + 1,  # Unique ID
#             'coach_name': str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else '',
#             'residential_area': str(row.iloc[1]).strip() if pd.notna(row.iloc[1]) else '',
#             'assigned_branch': str(row.iloc[2]).strip() if pd.notna(row.iloc[2]) else '',
#             'position': str(row.iloc[4]).strip() if pd.notna(row.iloc[4]) else '',
#             'status': str(row.iloc[5]).strip() if pd.notna(row.iloc[5]) else ''
#         }
        
#         # Add class capabilities as boolean columns (simplified names)
#         class_types = [
#             ('BearyTots', 9),
#             ('Jolly', 10), 
#             ('Bubbly', 11), 
#             ('Lively', 12), 
#             ('Flexi', 13),
#             ('Level_1', 14), 
#             ('Level_2', 15), 
#             ('Level_3', 16), 
#             ('Level_4', 17),
#             ('Advance', 18)  # This will combine L5, Advance, and Free
#         ]
        
#         for class_type, col_idx in class_types:
#             if col_idx < len(row) and pd.notna(row.iloc[col_idx]):
#                 can_teach = str(row.iloc[col_idx]).strip().lower() == 'yes'
#                 coach_info[class_type] = can_teach
#             else:
#                 coach_info[class_type] = False
        
#         # Combine Level 5 (col 18), Advance (col 19), and Free (col 20) into single 'Advance' category
#         level_5_can_teach = False
#         advance_can_teach = False
#         free_can_teach = False
        
#         if 18 < len(row) and pd.notna(row.iloc[18]):
#             level_5_can_teach = str(row.iloc[18]).strip().lower() == 'yes'
#         if 19 < len(row) and pd.notna(row.iloc[19]):
#             advance_can_teach = str(row.iloc[19]).strip().lower() == 'yes'
#         if 20 < len(row) and pd.notna(row.iloc[20]):
#             free_can_teach = str(row.iloc[20]).strip().lower() == 'yes'
        
#         # If coach can teach any of Level 5, Advance, or Free, mark as can teach Advance
#         coach_info['Advance'] = level_5_can_teach or advance_can_teach or free_can_teach
        
#         coaches_data.append(coach_info)
    
#     coaches_df = pd.DataFrame(coaches_data)
    
#     # 2. AVAILABILITY DATAFRAME - Day/time availability only
#     availability_data = []
    
#     for _, coach in coaches_df.iterrows():
#         row = data_rows[coach['coach_id'] - 1]  # Get original row data
        
#         full_off = str(row.iloc[6]).strip() if pd.notna(row.iloc[6]) else ''
#         half_off = str(row.iloc[7]).strip() if pd.notna(row.iloc[7]) else ''
#         restricted_times = str(row.iloc[8]).strip() if pd.notna(row.iloc[8]) else ''
        
#         coach_availability = parse_coach_availability(
#             coach['coach_id'], 
#             full_off, 
#             half_off, 
#             restricted_times
#         )
#         availability_data.extend(coach_availability)
    
#     availability_df = pd.DataFrame(availability_data)
    
#     return coaches_df, availability_df

# def parse_coach_availability(coach_id, full_off, half_off, restricted_times):
#     """
#     Parse coach availability into detailed day/time slots
#     """
#     availability = []
#     days = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']
#     time_periods = ['am', 'pm']
    
#     # Parse full off days
#     full_off_days = set()
#     if full_off:
#         full_off_text = full_off.lower()
#         for day in days:
#             if day.lower() in full_off_text:
#                 full_off_days.add(day)
    
#     # Parse half off days
#     half_off_periods = {}
#     if half_off:
#         half_off_text = half_off.lower()
#         for day in days:
#             if f"{day.lower()} am" in half_off_text:
#                 half_off_periods[day] = half_off_periods.get(day, []) + ['am']
#             if f"{day.lower()} pm" in half_off_text:
#                 half_off_periods[day] = half_off_periods.get(day, []) + ['pm']
    
#     # Parse restricted times
#     restricted_periods = parse_restricted_times(restricted_times)
    
#     # Generate availability for each day/period
#     availability_id = 1
#     for day in days:
#         for period in time_periods:
#             available = True
#             restriction_reason = None
            
#             # Check if fully off
#             if day in full_off_days:
#                 available = False
#                 restriction_reason = 'full_day_off'
            
#             # Check if half off
#             elif day in half_off_periods and period in half_off_periods[day]:
#                 available = False
#                 restriction_reason = 'half_day_off'
            
#             # Check if restricted
#             elif day in restricted_periods and period in restricted_periods[day]:
#                 available = False
#                 restriction_reason = 'time_restriction'
            
#             availability.append({
#                 'availability_id': availability_id,
#                 'coach_id': coach_id,
#                 'day': day,
#                 'period': period,
#                 'available': available,
#                 'restriction_reason': restriction_reason
#             })
#             availability_id += 1
    
#     return availability

# def parse_restricted_times(restricted_times):
#     """
#     Parse restricted time strings into structured data
#     """
#     restricted_periods = {}
#     days = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']
    
#     if not restricted_times:
#         return restricted_periods
    
#     restricted_text = restricted_times.lower()
    
#     import re
    
#     # Handle day ranges like "Mon - Thu am pm"
#     range_pattern = r'(\w+)\s*-\s*(\w+)\s+(am\s+pm|am|pm)'
#     ranges = re.findall(range_pattern, restricted_text)
    
#     for start_day, end_day, period in ranges:
#         start_idx = next((i for i, d in enumerate(days) if d.lower().startswith(start_day.lower())), -1)
#         end_idx = next((i for i, d in enumerate(days) if d.lower().startswith(end_day.lower())), -1)
        
#         if start_idx != -1 and end_idx != -1:
#             for i in range(start_idx, end_idx + 1):
#                 day = days[i]
#                 periods = ['am', 'pm'] if 'am pm' in period else [period.strip()]
#                 restricted_periods[day] = restricted_periods.get(day, []) + periods
    
#     # Handle individual days like "Mon am pm"
#     individual_pattern = r'(\w+)\s+(am\s+pm|am|pm)'
#     individuals = re.findall(individual_pattern, restricted_text)
    
#     for day_abbr, period in individuals:
#         day = next((d for d in days if d.lower().startswith(day_abbr.lower())), None)
#         if day:
#             periods = ['am', 'pm'] if 'am pm' in period else [period.strip()]
#             restricted_periods[day] = restricted_periods.get(day, []) + periods
    
#     return restricted_periods

# # Main execution
# coaches_df, availability_df = clean_coach_dataframe(df2)

# print("=== DATAFRAME STRUCTURES ===")
# print(f"\n1. COACHES_DF ({coaches_df.shape}):")
# print("Columns:", list(coaches_df.columns))

# print(f"\n2. AVAILABILITY_DF ({availability_df.shape}):")
# print("Columns:", list(availability_df.columns))

# # Show sample coach with capabilities
# print(f"\nSample coach record:")
# sample_coach = coaches_df.iloc[0].to_dict()
# for key, value in sample_coach.items():
#     print(f"  {key}: {value}")

# # Show class type columns specifically
# class_columns = ['BearyTots', 'Jolly', 'Bubbly', 'Lively', 'Flexi', 'Level_1', 'Level_2', 'Level_3', 'Level_4', 'Advance']
# print(f"\nClass capability columns: {class_columns}")

In [51]:
# coaches_df.to_csv('cleaned data/coaches_df.csv', index=False, encoding='utf-8')
# availability_df.to_csv('cleaned data/availability_df.csv', index=False, encoding='utf-8')

In [52]:
# availability_df

In [53]:
# capabilities_df